# Phase 1: Environment Setup
### 1.1 Library Imports
Loading required data science and visualization libraries.

In [32]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pylab as plt
import scipy as sp
import sklearn as sk
print('done!')

done!


# Phase 2: Processing Playlists Dimension (`dim_playlists`)
### 2.1 Data Loading & Initial Inspection
Loading the raw playlists dataset and viewing schema.

In [33]:
df = pd.read_csv('dim_playlists.csv')
df.head()

,playlist_id,playlist_name,published_at,total_videos
0,PLmur3Z0Afau49M9Qhc9ljxktYH58szH87,AI Architects,2026-04-02T16:07:19.491746Z,3
1,PLmur3Z0Afau6EJFQNsmCd0badJOW7vN9J,The Inspiration Lab,2025-11-20T19:00:23.715012Z,3
2,PLmur3Z0Afau6_ebbQaBXqwocwVN7y0_6y,Well Spent,2025-11-14T23:08:40.000088Z,2
3,PLmur3Z0Afau728_hvIL3xdqygK6lj27Eq,War Journal,2025-11-11T22:19:17.741199Z,0
4,PLmur3Z0Afau6OyqRXF96t97YxY-dYfv3L,Well Spent | The Podcast,2025-10-31T20:00:43.357812Z,2


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   playlist_id    51 non-null     object
 1   playlist_name  51 non-null     object
 2   published_at   51 non-null     object
 3   total_videos   51 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 1.7+ KB


### 2.2 Data Transformation: Datetime Parsing
Extracting Date and Time components from the ISO 8601 timestamp.

In [35]:
df['published_at'] = pd.to_datetime(df['published_at'])
df['published_date'] = df['published_at'].dt.date
df['published_date'] = pd.to_datetime(df['published_date'])
df['published_time'] = df['published_at'].dt.time
df.drop(columns=['published_at'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   playlist_id     51 non-null     object        
 1   playlist_name   51 non-null     object        
 2   total_videos    51 non-null     int64         
 3   published_date  51 non-null     datetime64[ns]
 4   published_time  51 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 2.1+ KB


# Phase 3: Processing Videos Dimension (`dim_videos`)
### 3.1 Data Loading & Shape Verification
Loading the raw videos dataset for dimensional modeling.

In [36]:
df_dim_vids = pd.read_csv('dim_videos.csv') 
df_dim_vids.shape

(7841, 26)

### 3.2 Data Transformation: Datetimes & Time Durations
Separating publish dates and converting video durations from seconds to a standard format.

In [37]:
df_dim_vids['published_at'] = pd.to_datetime(df_dim_vids['published_at'])
df_dim_vids['published_date']=df_dim_vids['published_at'].dt.date
df_dim_vids['published_date'] = pd.to_datetime(df_dim_vids['published_date'])
df_dim_vids['published_time']=df_dim_vids['published_at'].dt.time
df_dim_vids.drop(columns = ['published_at'], inplace = True)
df_dim_vids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7841 entries, 0 to 7840
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   video_id               7841 non-null   object        
 1   title                  7841 non-null   object        
 2   duration_seconds       7841 non-null   int64         
 3   view_count             7841 non-null   int64         
 4   like_count             7841 non-null   int64         
 5   comment_count          7841 non-null   int64         
 6   tags                   7702 non-null   object        
 7   is_hd                  7841 non-null   bool          
 8   licensed_content       7841 non-null   bool          
 9   made_for_kids          7841 non-null   bool          
 10  audio_language         7841 non-null   object        
 11  region_allowed         0 non-null      float64       
 12  region_blocked         0 non-null      float64       
 13  con

In [38]:
df_dim_vids['duration'] = pd.to_datetime(df_dim_vids['duration_seconds'], unit='s').dt.strftime('%H:%M:%S')

### 3.3 Feature Selection: Dropping Irrelevant Columns
Removing geospatial and blocked region metadata not needed for our current analysis.

In [39]:
df_dim_vids.drop(columns=['region_allowed',
                          'region_blocked', 
                          'latitude', 
                          'longitude', 
                          'recording_date'],
                           inplace = True)
df_dim_vids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7841 entries, 0 to 7840
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   video_id               7841 non-null   object        
 1   title                  7841 non-null   object        
 2   duration_seconds       7841 non-null   int64         
 3   view_count             7841 non-null   int64         
 4   like_count             7841 non-null   int64         
 5   comment_count          7841 non-null   int64         
 6   tags                   7702 non-null   object        
 7   is_hd                  7841 non-null   bool          
 8   licensed_content       7841 non-null   bool          
 9   made_for_kids          7841 non-null   bool          
 10  audio_language         7841 non-null   object        
 11  content_rating         7841 non-null   object        
 12  privacy_status         7841 non-null   object        
 13  lic

### 3.4 Data Profiling: Duplicates Assessment
Checking for duplicated records in the API payload.

In [40]:
df_dim_vids.duplicated().sum()

np.int64(0)

### 3.5 Data Cleaning: Standardizing Categorical Features
Extracting nested dictionary values (like `ytAgeRestricted`) and standardizing language codes.

In [41]:
df_dim_vids["content_rating"].unique().tolist()

['{}', "{'ytRating': 'ytAgeRestricted'}"]

In [42]:
df_dim_vids[df_dim_vids['content_rating'] == "{'ytRating': 'ytAgeRestricted'}"]

,video_id,title,duration_seconds,view_count,like_count,comment_count,tags,is_hd,licensed_content,made_for_kids,...,license,embeddable,public_stats_viewable,topic_categories,has_captions,primary_hashtag,content_type,published_date,published_time,duration
2037,MzU_8jQ_4zs,This farmer uses gasoline to make coca paste #...,59,645920,20337,451,"Business Insider, Business News",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Lifestyle_(socio...,False,NaN,Short,2023-08-25,16:18:52,00:00:59
2334,QxbkaqfL_bI,Why People Risk Their Lives To Kill Bulls That...,726,212509,2520,1123,"Insider, News, bull fighting, spain, risky bus...",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Society,False,#Bulls,Standard Video,2023-02-04,17:00:28,00:12:06
2433,SD66p3XE_TE,Every Piece Of Gear In A US Army Combat Medic'...,792,156795,3890,233,"Business Insider, Business News, Loadout, Army...",True,True,False,...,youtube,True,True,"https://en.wikipedia.org/wiki/Military, https:...",True,#Army,Standard Video,2022-11-10,16:00:13,00:13:12
2521,XM94wRqgMes,Biggest challenges for the #cannabis industry ...,43,47708,1845,59,"Business Insider, Business News",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Lifestyle_(socio...,False,NaN,Short,2022-08-03,18:00:10,00:00:43
2550,gDEVO02KIa0,How Bread Became Unaffordable Across The World...,701,1984786,22969,3173,"Business Insider, Business News, Big Business,...",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Society,True,#GrainShortage,Standard Video,2022-06-16,21:00:33,00:11:41
2987,uadGTl8Hwss,Why Online Sex Work Has Risen During The Pandemic,411,111613,561,115,"Insider, News, sex, sex work, only fans, femal...",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Lifestyle_(socio...,True,#Sex,Standard Video,2020-08-15,16:00:13,00:06:51
3041,wTLAfT4w1zQ,What It's Like Being A Funeral Director During...,346,385415,5474,478,"Business Insider, Business News, funerals, cor...",True,True,False,...,youtube,True,True,"https://en.wikipedia.org/wiki/Health, https://...",True,#Coronavirus,Standard Video,2020-04-07,21:28:46,00:05:46
3107,xvY_TcBn2Gs,Why Full-Spectrum CBD Oil is So Expensive | So...,397,344701,6133,530,"Business News, CBD, CBD oil, Business Insider,...",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Health,True,#CBD,Standard Video,2019-12-21,16:00:08,00:06:37
3184,xGYA5hx6wP4,What It's Like Inside A Canadian Marijuana Gre...,208,399162,6452,649,"Business Insider, Business News, cannabis, mar...",True,True,False,...,youtube,True,True,https://en.wikipedia.org/wiki/Society,False,#Canada,Standard Video,2019-09-05,15:00:10,00:03:28
3690,uy-VZHqwWNM,The Racist Origins of Marijuana Prohibition,402,102475,2540,377,"Business Insider, Cannabis, Marijuana, Laws, L...",True,True,False,...,youtube,True,True,"https://en.wikipedia.org/wiki/Politics, https:...",True,NaN,Standard Video,2018-03-01,15:31:17,00:06:42


In [43]:

df_dim_vids['content_rating'].replace({'{}' : 'not age-restricted', "{'ytRating': 'ytAgeRestricted'}" : "age-restricted" }, inplace = True)

In [44]:
for col in df_dim_vids.columns :
    if df_dim_vids[col].dtype == 'object':
        name = f'{col}_uniques'
        liste = df_dim_vids[col].unique().tolist()
        print(f"{name} : {liste}\n")

video_id_uniques : ['WruHZnB_srw', 'BDqhOhr-orw', 'r3p0R9dlKww', 'Gv8mItBLZ5Y', 'dKoIepQtPO8', 'dp5QFSjloNM', '1Qp1yaSpfx8', 'TM47c__WOfQ', 'bG6bXMhy0Ps', 'x3wVNJlem7U', 'rWSL0v-rmak', 'OCGQIAk8zIM', 'PJqh69mNTtM', '_rcRea7DG10', 'g4oIYbI9tog', 'sA_U4DhYmbc', 'QQvvO8Mqk1w', 'yqBwFlECOXc', 'QG6cEqow99Y', 'KmE6OPhpLjw', 'GtjkUge5aew', 'CngqU51dsJU', 's80mBDNJ5lo', 'oslN3LtaBqI', 'SIjMGjKA9Rs', 'bzDonR6Pfo4', 'I6LPhS7yH-c', 'Pf8eNoGqut0', '9JLKfP5uJ9E', 'rCjtlBmKkMU', 'shyDrSrHcwo', 'ma5IsOYGxrs', 'pOi_AQ9F_Mo', '2-s8P58EwTY', 'ZyGH4eUXAX4', 'nD0bNdRu0w8', 'KH6O-MuUdB8', 'GG7j5PqX4UY', 'yxF_RnU400k', 'dfN3BQt6i98', 'ZCs_Z86qvYU', 'jCkvBQWhrNA', 'ENRrTJdlyg8', 'lRUS-7P0a4k', 'uGkwc-JWdYw', 'ydqkCytz9GI', 'BsEfPvIrVCc', 'RljBVCnt9AQ', '5MIn6v9h0oQ', 'Qytqug0QgKg', '1d_KSbE2LX8', 'aMtaVPjiOqw', 'NwTg6HjkEgQ', 'C3VjFb1_ct4', 'DUgesPqAztc', 'jvdo-c0Pa1M', 'vSYgZG9UNN4', '1YDQiiBKgTQ', '9djOjC1Uszc', 'FfqVuyF8wnY', 'ULPQZsj0J2Y', 'hohnMVXn91E', 'HjSQ10zDB0k', 'T_RbZxmafrE', 'bvqzzusB-_M', '6_aK

In [45]:
df_dim_vids = df_dim_vids.drop(columns = ['privacy_status', 'license'])

In [46]:
df_dim_vids['audio_language'].replace({'en-US' : 'English (United States)',
                                       'en' : 'English',
                                       'zh' : 'Chinese',
                                       'ja' : 'Japanese',
                                       'ar' : 'Arabic'}, inplace = True)

### 3.6 Data Cleaning: Boolean Mapping
Converting boolean `True/False` values into descriptive string categories for easier dashboarding.

In [47]:
labels = {'has_captions' : ['has caption', 'has no caption'],
          'is_hd' : ['HD', 'not HD' ],
          'licensed_content' : ['licensed', 'unlicensed'],
          'made_for_kids' : ['for kids', 'not for kids'],
          'embeddable' : ['embeddable', 'not embeddable'],
          'public_stats_viewable' : ['stats visible', 'stats hidden']}

for col in labels.keys() :
    df_dim_vids[col].replace({True : labels[col][0],
                              False : labels[col][1]},
                              inplace = True)

In [48]:
for col in df_dim_vids.columns :
    if df_dim_vids[col].dtype == 'object':
        name = f'{col}_uniques'
        liste = df_dim_vids[col].unique().tolist()
        print(f"{name} : {liste}\n")

video_id_uniques : ['WruHZnB_srw', 'BDqhOhr-orw', 'r3p0R9dlKww', 'Gv8mItBLZ5Y', 'dKoIepQtPO8', 'dp5QFSjloNM', '1Qp1yaSpfx8', 'TM47c__WOfQ', 'bG6bXMhy0Ps', 'x3wVNJlem7U', 'rWSL0v-rmak', 'OCGQIAk8zIM', 'PJqh69mNTtM', '_rcRea7DG10', 'g4oIYbI9tog', 'sA_U4DhYmbc', 'QQvvO8Mqk1w', 'yqBwFlECOXc', 'QG6cEqow99Y', 'KmE6OPhpLjw', 'GtjkUge5aew', 'CngqU51dsJU', 's80mBDNJ5lo', 'oslN3LtaBqI', 'SIjMGjKA9Rs', 'bzDonR6Pfo4', 'I6LPhS7yH-c', 'Pf8eNoGqut0', '9JLKfP5uJ9E', 'rCjtlBmKkMU', 'shyDrSrHcwo', 'ma5IsOYGxrs', 'pOi_AQ9F_Mo', '2-s8P58EwTY', 'ZyGH4eUXAX4', 'nD0bNdRu0w8', 'KH6O-MuUdB8', 'GG7j5PqX4UY', 'yxF_RnU400k', 'dfN3BQt6i98', 'ZCs_Z86qvYU', 'jCkvBQWhrNA', 'ENRrTJdlyg8', 'lRUS-7P0a4k', 'uGkwc-JWdYw', 'ydqkCytz9GI', 'BsEfPvIrVCc', 'RljBVCnt9AQ', '5MIn6v9h0oQ', 'Qytqug0QgKg', '1d_KSbE2LX8', 'aMtaVPjiOqw', 'NwTg6HjkEgQ', 'C3VjFb1_ct4', 'DUgesPqAztc', 'jvdo-c0Pa1M', 'vSYgZG9UNN4', '1YDQiiBKgTQ', '9djOjC1Uszc', 'FfqVuyF8wnY', 'ULPQZsj0J2Y', 'hohnMVXn91E', 'HjSQ10zDB0k', 'T_RbZxmafrE', 'bvqzzusB-_M', '6_aK

### 3.7 Handling Missing Values (Imputation)
Filling NaNs in text arrays, hashtags, and assigned Wikipedia topics.

In [49]:
df_dim_vids['primary_hashtag'] = df_dim_vids['primary_hashtag'].fillna('No primary hashtag')
df_dim_vids['tags'] = df_dim_vids['tags'].fillna('No tags mentioned')

In [50]:
df_dim_vids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7841 entries, 0 to 7840
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   video_id               7841 non-null   object        
 1   title                  7841 non-null   object        
 2   duration_seconds       7841 non-null   int64         
 3   view_count             7841 non-null   int64         
 4   like_count             7841 non-null   int64         
 5   comment_count          7841 non-null   int64         
 6   tags                   7841 non-null   object        
 7   is_hd                  7841 non-null   object        
 8   licensed_content       7841 non-null   object        
 9   made_for_kids          7841 non-null   object        
 10  audio_language         7841 non-null   object        
 11  content_rating         7841 non-null   object        
 12  embeddable             7841 non-null   object        
 13  pub

In [51]:
df_dim_vids.rename(columns={'topic_categories': 'assigned_wiki_topic'}, inplace = True)
df_dim_vids['assigned_wiki_topic'] = df_dim_vids['assigned_wiki_topic'].fillna('No wiki topic assigned')
df_dim_vids['assigned_wiki_topic'].unique().tolist()


['https://en.wikipedia.org/wiki/Business, https://en.wikipedia.org/wiki/Society',
 'https://en.wikipedia.org/wiki/Food, https://en.wikipedia.org/wiki/Lifestyle_(sociology)',
 'https://en.wikipedia.org/wiki/Military, https://en.wikipedia.org/wiki/Society',
 'https://en.wikipedia.org/wiki/Politics',
 'https://en.wikipedia.org/wiki/Politics, https://en.wikipedia.org/wiki/Society',
 'No wiki topic assigned',
 'https://en.wikipedia.org/wiki/Health, https://en.wikipedia.org/wiki/Society',
 'https://en.wikipedia.org/wiki/Lifestyle_(sociology)',
 'https://en.wikipedia.org/wiki/Food, https://en.wikipedia.org/wiki/Hobby, https://en.wikipedia.org/wiki/Lifestyle_(sociology)',
 'https://en.wikipedia.org/wiki/Technology',
 'https://en.wikipedia.org/wiki/Society',
 'https://en.wikipedia.org/wiki/Lifestyle_(sociology), https://en.wikipedia.org/wiki/Pet',
 'https://en.wikipedia.org/wiki/Military',
 'https://en.wikipedia.org/wiki/Military, https://en.wikipedia.org/wiki/Politics, https://en.wikipedia.org

### 3.8. Exporting the current df_dim_vids as CSV for future fact table making :

In [52]:
df_fact_vids = df_dim_vids[['video_id', 'tags', 'assigned_wiki_topic']]
df_fact_vids.to_csv('fact_vids.csv', index=False)

### 3.9. Preparing df_dim_vids for exporting by removing the column which belong to fact table for more preparation :

In [53]:
df_dim_vids.drop(columns ={'tags', 'assigned_wiki_topic'}, inplace = True)
df_dim_vids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7841 entries, 0 to 7840
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   video_id               7841 non-null   object        
 1   title                  7841 non-null   object        
 2   duration_seconds       7841 non-null   int64         
 3   view_count             7841 non-null   int64         
 4   like_count             7841 non-null   int64         
 5   comment_count          7841 non-null   int64         
 6   is_hd                  7841 non-null   object        
 7   licensed_content       7841 non-null   object        
 8   made_for_kids          7841 non-null   object        
 9   audio_language         7841 non-null   object        
 10  content_rating         7841 non-null   object        
 11  embeddable             7841 non-null   object        
 12  public_stats_viewable  7841 non-null   object        
 13  has

### 3.10 Final Validation & Export
Reviewing the cleaned dataset and exporting to a clean CSV file.

In [54]:
df_dim_vids.head()

,video_id,title,duration_seconds,view_count,like_count,comment_count,is_hd,licensed_content,made_for_kids,audio_language,content_rating,embeddable,public_stats_viewable,has_captions,primary_hashtag,content_type,published_date,published_time,duration
0,WruHZnB_srw,#Tech #stocks drop after reports say #OpenAI f...,71,17985,429,23,HD,licensed,not for kids,English (United States),not age-restricted,embeddable,stats visible,has no caption,No primary hashtag,Standard Video,2026-04-28,19:46:26,00:01:11
1,BDqhOhr-orw,The Rise And Fall Of 7 American Businesses,12634,26268,408,24,HD,licensed,not for kids,English (United States),not age-restricted,embeddable,stats visible,has no caption,#USbusinesses,Standard Video,2026-04-28,19:45:00,03:30:34
2,r3p0R9dlKww,"Priced at $209M, the MQ-25 #Stingray is design...",62,73649,941,55,HD,licensed,not for kids,English (United States),not age-restricted,embeddable,stats visible,has no caption,No primary hashtag,Standard Video,2026-04-27,22:14:47,00:01:02
3,Gv8mItBLZ5Y,From alleged hairdryer hacking to insider #tra...,84,6282,76,3,HD,licensed,not for kids,English (United States),not age-restricted,embeddable,stats visible,has no caption,No primary hashtag,Standard Video,2026-04-27,19:47:02,00:01:24
4,dKoIepQtPO8,Footage shows moments after security detains #...,54,7491,88,11,HD,licensed,not for kids,English (United States),not age-restricted,embeddable,stats visible,has no caption,No primary hashtag,Short,2026-04-26,15:49:48,00:00:54


In [55]:
df_dim_vids.to_csv('dim_vids_clean.csv', index=False)

# Phase 4: Processing the Fact Table
### 4.1 Fact Table Initialization
Preparing the transactional fact metrics (views, likes, comments).

In [56]:
df_fact_vids = pd.read_csv('fact_vids.csv')

### 4.2. Explode all assigned wikipidia URLs of topics to have one in each row for more better practices:

In [57]:
df_fact_vids['assigned_wiki_topic'] = df_fact_vids['assigned_wiki_topic'].str.split(', ')
df_fact_vids = df_fact_vids.explode('assigned_wiki_topic')

In [58]:
df_fact_vids.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12666 entries, 0 to 7840
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             12666 non-null  object
 1   tags                 12666 non-null  object
 2   assigned_wiki_topic  12666 non-null  object
dtypes: object(3)
memory usage: 395.8+ KB


### 4.3. remove the URLs path and leave only the assigned topic part from wikipedia URLs :

In [59]:
df_fact_vids['assigned_wiki_topic'] = df_fact_vids['assigned_wiki_topic'].str.split('/').str[-1].str.strip()


### 4.4. Explode all tags to have one in each row for more better practices :

In [60]:
df_fact_vids['tags'] = df_fact_vids['tags'].str.split(', ')
df_fact_vids = df_fact_vids.explode('tags')
df_fact_vids['tags'] = df_fact_vids['tags'].str.strip()

In [61]:
df_fact_vids.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93873 entries, 0 to 7840
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             93873 non-null  object
 1   tags                 93873 non-null  object
 2   assigned_wiki_topic  93873 non-null  object
dtypes: object(3)
memory usage: 2.9+ MB


In [63]:
df_fact_vids.to_csv('fact_vids_clean.csv', index = False)